In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests # for the owid data

# the notebook lives in the "notebooks" folder, the data in "../data"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

### this is the code tha the owid website uses for pandas python import

In [2]:
import requests

# Fetch the data.
owid_gini = pd.read_csv("https://ourworldindata.org/grapher/gini-coefficient-wb.csv?v=1&csvType=full&useColumnShortNames=true&survey_comparability=no_spells", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata_gini = requests.get("https://ourworldindata.org/grapher/gini-coefficient-wb.metadata.json?v=1&csvType=full&useColumnShortNames=true&survey_comparability=no_spells").json()

# Fetch the data.
owid_gdp = pd.read_csv("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata_gdp = requests.get("https://ourworldindata.org/grapher/gdp-per-capita-worldbank.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

#make sure names are ziped to country code
name_to_iso = dict(zip(owid_gdp["entity"], owid_gdp["code"]))      
name_to_iso_gini = dict(zip(owid_gini["entity"], owid_gini["code"]))  
# name_to_iso.update({
#     "Korea": "KOR", "Republic of Korea": "KOR", "Slovak Republic": "SVK", "Türkiye": "TUR",
#     "England (UK)": "GBR",              # the health survey only covers England -> used for the UK
#     "Flemish Region (Belgium)": "BEL",  # ... and only Flanders -> used for Belgium
# })

In [5]:
p = pd.read_csv(RAW / "OECD_well_being_2022_2023.csv")
p = p[p["Measure"].str.startswith("Student")]      # the 3 PISA subjects (maths, reading, science)
p = p[p["TIME_PERIOD"] == 2022]                     # PISA 2022

# all scores: one row per country, one column per (parents' education, subject)
pisa_by_group = p.pivot_table(index="REF_AREA", columns=["Education level", "Measure"], values="OBS_VALUE")

# main PISA score: students whose parents have secondary education, mean of the 3 subjects
pisa = p[p["Education level"] == "Secondary education"].groupby("REF_AREA")["OBS_VALUE"].mean()
pisa.name = "pisa"

pisa_table = (
    pisa.reset_index()
        .rename(columns={
            "REF_AREA": "country code",
            "pisa": "pisa score"
        })
)

pisa_table.head(10)

,country code,pisa score
0,ARG,363.962678
1,AUS,452.286124
2,AUT,444.306710
3,BEL,431.933898
4,BGR,339.616696
5,BRA,367.198359
6,CAN,445.874590
7,CHE,450.231146
8,CHL,389.030916
9,COL,389.507523


### 2b. Average years of education (detailed version)
The OECD file gives the % of adults (25–64) at each education level. We give every level a typical number of years
and take the weighted average:

| Level | OECD code | Years |
|---|---|---|
| Below upper secondary | ISCED11A_0T2 | 9 |
| Upper secondary / post-secondary non-tertiary | ISCED11A_3_4 | 12.5 |
| Short-cycle tertiary (e.g. 2-year college) | ISCED11A_5 | 14 |
| Bachelor's | ISCED11A_6 | 15.5 |
| Master's | ISCED11A_7 | 17.5 |
| Doctoral | ISCED11A_8 | 21 |

edu_years = Σ (share of level × years of level) / 100



In [7]:
e = pd.read_csv(RAW / "OECD.EDU_2025.csv")
e = e[e["REF_AREA"] != "OECD"]                                   # remove the OECD average (not a country)

# one row per country, one column per education level (values = % of adults 25-64)
shares = e.pivot_table(index="REF_AREA", columns="ATTAINMENT_LEV", values="OBS_VALUE")

# typical years of schooling for each level (change these numbers if you want other assumptions)
YEARS = {
    "ISCED11A_0T2": 9,      # below upper secondary
    "ISCED11A_3_4": 12.5,   # upper secondary / post-secondary non-tertiary
    "ISCED11A_5":   14,     # short-cycle tertiary
    "ISCED11A_6":   15.5,   # bachelor's
    "ISCED11A_7":   17.5,   # master's
    "ISCED11A_8":   21,     # doctoral
}

# Rule 1: a missing level counts as 0 %.
#   Some countries report e.g. doctoral degrees inside the master's figure ("included in another category"),
#   or the level does not exist in the country - in both cases 0 is correct.
s = shares[list(YEARS)].fillna(0)

# Rule 2: if a country only reports tertiary as a total (e.g. Colombia), the part of tertiary that is not
#   split into short-cycle/bachelor's/master's/doctoral gets an average of 16 years.
tertiary_detail = s[["ISCED11A_5", "ISCED11A_6", "ISCED11A_7", "ISCED11A_8"]].sum(axis=1)
s["tertiary_unspecified"] = (shares["ISCED11A_5T8"] - tertiary_detail).clip(lower=0)
YEARS["tertiary_unspecified"] = 16

# check: the shares of a country should add up to ~100 %; drop countries with incomplete data
total = s.sum(axis=1)
print("countries dropped (shares do not add up to ~100 %):", list(total[~total.between(95, 105)].index))
s = s[total.between(95, 105)]
s = s.div(s.sum(axis=1), axis=0) * 100                          # make the shares add up to exactly 100 %

# weighted average: sum over levels of (share x years), divided by 100
edu_years = sum(s[level] * years for level, years in YEARS.items()) / 100
edu_years.name = "edu_years"
#edu_years.sort_values().round(2)

countries dropped (shares do not add up to ~100 %): ['PER']


In [8]:
edu_table = (
    s.rename(columns={
        "ISCED11A_0T2": "below_secondary_pct",
        "ISCED11A_3_4": "upper_secondary_pct",
        "ISCED11A_5": "short_cycle_tertiary_pct",
        "ISCED11A_6": "bachelor_pct",
        "ISCED11A_7": "master_pct",
        "ISCED11A_8": "doctoral_pct",
        "tertiary_unspecified": "tertiary_unspecified_pct"
    })
)

edu_table["average years"] = edu_years

edu_table = (
    edu_table.reset_index()
             .rename(columns={"REF_AREA": "country code"})
             .round(2)
)

edu_table.head(10)

ATTAINMENT_LEV,country code,below_secondary_pct,upper_secondary_pct,short_cycle_tertiary_pct,bachelor_pct,master_pct,doctoral_pct,tertiary_unspecified_pct,average years
0,AUS,11.93,33.69,11.43,31.32,10.03,1.61,0.00,13.83
1,AUT,13.21,49.19,14.91,6.39,15.13,1.18,0.00,13.31
2,BEL,15.07,37.48,1.22,25.03,20.05,1.16,0.00,13.84
3,BGR,13.03,51.30,0.00,10.77,24.32,0.58,0.00,13.63
4,CAN,6.57,29.49,24.55,25.03,14.35,0.00,0.00,14.11
5,CHE,13.90,39.86,0.00,24.97,17.96,3.30,0.00,13.94
6,COL,34.60,35.91,0.00,0.00,0.00,0.00,29.49,12.32
7,CRI,50.34,21.43,7.48,17.57,3.13,0.06,0.00,11.54
8,CZE,5.59,65.68,0.15,7.87,19.79,0.92,0.00,13.61
9,DEU,14.05,50.46,0.57,20.16,12.73,2.03,0.00,13.43


In [9]:
o = owid_gdp[(owid_gdp["year"] == 2023) & owid_gdp["code"].notna()]            # rows without a code are regions (Europe, World ...)
income = o.set_index("code")["ny_gdp_pcap_pp_kd"] / 1000
income.name = "income"

g = owid_gini.rename(columns={owid_gini.columns[-1]: "gini"})                       # the last column holds the Gini value (0-1)
g = g[g["year"] <= 2023].sort_values("year")
gini = g.dropna(subset=["code"]).groupby("code")["gini"].last() * 100   

In [11]:
w = pd.read_excel(RAW / "happyness-data.xlsx")
w = w[w["Year"] == 2023]
w["iso3"] = w["Country name"].map(name_to_iso)
happiness = w.dropna(subset=["iso3"]).set_index("iso3")["Life evaluation (3-year average)"]
happiness.name = "happiness"